# Step 3 — Lag-Llama World Model (Fine-Tuned, Rolling Re-Trained)

**Changes applied in this pass (per team requirements):**

1. **Fine-tuned, not zero-shot (Cell 9).** The estimator now calls `.train()` on our own
   2018-2022 data (validated on 2023-2024) starting from the pretrained `lag-llama.ckpt`
   weights, instead of wrapping the untouched checkpoint into a predictor directly.
2. **Rolling re-training (Cell 11).** During the TEST-period walk-forward loop, the model is
   periodically re-fine-tuned on an expanding window of all data available up to that point
   (every `ROLLING_RETRAIN_EVERY` rebalance dates), so it keeps adapting as the market moves
   into new regimes instead of forecasting the whole test period with one static fit.
3. **Directional-accuracy confidence, not variance/MSE (Cells 9 & 11).** `Confidence` is now
   derived from how often the model's forecast direction (up/down) matched what actually
   happened on a held-out validation window, re-scored at every rolling re-train — replacing
   the old `1 / (1 + 100 * forecast_variance)` formula.
4. **Global dampening + cap (Cells 9 & 11).** Every confidence score is multiplied by 0.5 and
   then hard-capped at 0.75, via `confidence_from_hitrate()`.
5. **+/-1.5x historical monthly std clip (Cell 9 & 11).** `Expected_Return` is still clipped to
   +/-1.5x each asset's historical monthly return std — now computed directly from our 2018-2024
   data in Cell 9 instead of a hardcoded dict, so the numbers are traceable.

Everything below Cell 8 (checkpoint download) was rewritten to support this; Cells 0-7
(environment setup, data loading/cleaning, GluonTS formatting) are unchanged.


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("WARNING: No GPU found. Go to Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# Install from GitHub directly (not PyPI)
!pip install "git+https://github.com/time-series-foundation-models/lag-llama.git" -q
!pip install huggingface_hub "gluonts[torch]" pyportfolioopt -q

print("All libraries installed successfully!")

In [ ]:
import urllib.request

urls = [
    "https://huggingface.co",
    "https://github.com",
]

for url in urls:
    try:
        code = urllib.request.urlopen(url, timeout=10).getcode()
        print(f"{url} -> accessible (code: {code})")
    except Exception as e:
        print(f"{url} -> BLOCKED ({e})")

In [ ]:
# ============================================================
# Cell 4 — NumPy Reinstall + Safe Auto Restart (Fixed)
# ============================================================

import subprocess
import sys

# --- Step 1: Reinstall NumPy ---
print("⬇️  Reinstalling NumPy 1.26.4...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "numpy==1.26.4", "-q", "--force-reinstall"],
    capture_output=True,
    text=True
)

# --- Step 2: Verify installation ---
import numpy as np_check
print(f"✅ NumPy version confirmed: {np_check.__version__}")
print("")
print("=" * 50)
print("✅ NumPy Reinstall Complete!")
print("🔄 Runtime restarting in 5 seconds...")
print("📌 After restart → Run from Cell 5 onwards!")
print("=" * 50)

# --- Step 3: Countdown ---
import time
for i in range(5, 0, -1):
    print(f"   Restarting in {i}...")
    time.sleep(1)

print("🔄 Restarting now...")

# --- Step 4: Fixed restart method ---
import IPython
app = IPython.Application.instance()
app.kernel.do_shutdown(restart=True)

In [ ]:
# ============================================================
# Cell 5 — Load Dataset from Team GitHub Repository
# ============================================================

import pandas as pd
import numpy as np
from gluonts.dataset.common import ListDataset

# --- Load V2 dataset from updated GitHub location ---
url = "https://raw.githubusercontent.com/FM4056-GPT-Driven/Project/main/data/project8_dataset_v2.csv"

df = pd.read_csv(url)

print("✅ CSV Loaded from GitHub!")
print(f"   URL     : {url}")
print(f"   Shape   : {df.shape}")
print(f"   Columns : {list(df.columns)}")
print(df.head(3))

In [ ]:
# ============================================================
# Cell 6 — Extract, Clean and Split Dataset
# ============================================================

# --- Step 1: Parse date column ---
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# --- Step 2: Extract required columns ---
# POLICY_EVENT removed — handled by Llama-3 in Step 5
close_cols = ['Date', 'EURUSD_Close', 'GBPUSD_Close', 'GOLD_Close', 'OIL_Close',
              'VIX', 'USD_INDEX', 'US10Y_YIELD']

available_cols = [c for c in close_cols if c in df.columns]
missing_cols   = [c for c in close_cols if c not in df.columns]

print(f"✅ Available columns : {available_cols}")
print(f"⚠️  Missing columns  : {missing_cols}")

# --- Step 3: Build cleaned DataFrame ---
df_clean = df[available_cols].copy()
price_cols = [c for c in available_cols if c != 'Date']
df_clean = df_clean.dropna(subset=price_cols, how='all')

# --- Step 4: Train/Validation/Test Split ---
df_train = df_clean[df_clean['Date'] < '2023-01-01'].copy()
df_val   = df_clean[(df_clean['Date'] >= '2023-01-01') &
                    (df_clean['Date'] <  '2025-01-01')].copy()
df_test  = df_clean[df_clean['Date'] >= '2025-01-01'].copy()

print(f"\n✅ Cleaned shape : {df_clean.shape}")
print(f"\n✅ Train/Val/Test Split:")
print(f"   Train      : {df_train['Date'].min().date()} → {df_train['Date'].max().date()} ({len(df_train)} days)")
print(f"   Validation : {df_val['Date'].min().date()} → {df_val['Date'].max().date()} ({len(df_val)} days)")
print(f"   Test       : {df_test['Date'].min().date()} → {df_test['Date'].max().date()} ({len(df_test)} days)")

In [ ]:
# ============================================================
# Cell 7 — GluonTS Format + Macro Covariate Scaling
# ============================================================

import numpy as np
from gluonts.dataset.common import ListDataset
from sklearn.preprocessing import StandardScaler

# --- Step 1: Define all global variables ---
asset_cols        = ['EURUSD_Close', 'GBPUSD_Close', 'GOLD_Close', 'OIL_Close']
macro_cols        = ['VIX', 'USD_INDEX', 'US10Y_YIELD']
prediction_length = 30
freq              = "B"

# --- Step 2: Fill NaN values ---
for split_df in [df_train, df_val, df_test, df_clean]:
    split_df[macro_cols] = split_df[macro_cols].ffill()
    split_df[macro_cols] = split_df[macro_cols].bfill()
print("✅ Macro NaN values filled!")

# --- Step 3: Fit StandardScaler on TRAIN only ---
# Important: fit on train, transform all splits
scaler = StandardScaler()
scaler.fit(df_train[macro_cols])
print("✅ StandardScaler fitted on Train data!")

# --- Step 4: Transform all splits ---
macro_train_scaled = scaler.transform(df_train[macro_cols]).T.astype(np.float32)
macro_val_scaled   = scaler.transform(df_val[macro_cols]).T.astype(np.float32)
macro_test_scaled  = scaler.transform(df_test[macro_cols]).T.astype(np.float32)

print(f"\n✅ Scaled macro covariate shapes:")
print(f"   Train      : {macro_train_scaled.shape}")
print(f"   Validation : {macro_val_scaled.shape}")
print(f"   Test       : {macro_test_scaled.shape}")

# --- Step 5: Print scaling stats ---
print(f"\n✅ Scaling stats (Train):")
for i, col in enumerate(macro_cols):
    print(f"   {col}:")
    print(f"      Mean : {scaler.mean_[i]:.4f}")
    print(f"      Std  : {scaler.scale_[i]:.4f}")

# --- Step 6: Create GluonTS ListDatasets ---
train_data = ListDataset(
    [
        {
            "start"            : df_train['Date'].iloc[0],
            "target"           : df_train[col].values.astype(np.float32),
            "feat_dynamic_real": macro_train_scaled
        }
        for col in asset_cols
    ],
    freq=freq
)

val_data = ListDataset(
    [
        {
            "start"            : df_val['Date'].iloc[0],
            "target"           : df_val[col].values.astype(np.float32),
            "feat_dynamic_real": macro_val_scaled
        }
        for col in asset_cols
    ],
    freq=freq
)

test_data = ListDataset(
    [
        {
            "start"            : df_test['Date'].iloc[0],
            "target"           : df_test[col].values.astype(np.float32),
            "feat_dynamic_real": macro_test_scaled
        }
        for col in asset_cols
    ],
    freq=freq
)

print("\n✅ GluonTS ListDatasets created with scaled covariates!")
print(f"   Assets           : {asset_cols}")
print(f"   Macro covariates : {macro_cols}")
print(f"   Scaling method   : StandardScaler (mean=0, std=1)")
print(f"   Prediction length: {prediction_length} days")

# --- Step 7: Validate ---
print("\n✅ Validation (Train):")
for i, (col, entry) in enumerate(zip(asset_cols, train_data)):
    print(f"   [{i+1}] {col}")
    print(f"        target shape           : {entry['target'].shape}")
    print(f"        feat_dynamic_real shape: {entry['feat_dynamic_real'].shape}")

In [ ]:
# ============================================================
# Cell 8 — Download Lag-Llama Checkpoint
# ============================================================

import torch
import functools
from huggingface_hub import hf_hub_download
from gluonts.torch.distributions.studentT import StudentTOutput

# --- Step 1: Allow StudentTOutput as safe global ---
torch.serialization.add_safe_globals([StudentTOutput])
print("✅ Safe globals configured!")

# --- Step 2: Download checkpoint ---
print("⬇️  Downloading Lag-Llama checkpoint...")
ckpt_path = hf_hub_download(
    repo_id  = "time-series-foundation-models/Lag-Llama",
    filename = "lag-llama.ckpt"
)
print(f"✅ Checkpoint downloaded!")
print(f"   Path : {ckpt_path}")

# --- Step 3: Patch torch.load safely ---
original_load = torch.load.__wrapped__ if hasattr(torch.load, '__wrapped__') else torch.load

@functools.wraps(original_load)
def safe_load(f, *args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return original_load(f, *args, **kwargs)

torch.load = safe_load
print("✅ torch.load safely configured!")

In [ ]:
# ============================================================
# Cell 9 — Fine-Tune Lag-Llama on Our Own Data (Requirement #1: no longer zero-shot)
# ============================================================

import torch
import numpy as np
import pandas as pd
from gluonts.dataset.common import ListDataset
from sklearn.preprocessing import StandardScaler
from lag_llama.gluon.estimator import LagLlamaEstimator

# --- Step 0: Define variables ---
prediction_length = 30
context_length    = 64
asset_cols        = ['EURUSD_Close', 'GBPUSD_Close', 'GOLD_Close', 'OIL_Close']
macro_cols        = ['VIX', 'USD_INDEX', 'US10Y_YIELD']

# --- Step 1: Setup device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")

# --- Step 2: Reusable estimator builder -- always initializes from the pretrained lag-llama.ckpt
#     (ckpt_path), so every call below is a FINE-TUNE starting from those weights, never a
#     from-scratch train. Kept as a function so Cell 11's rolling re-training (Requirement #2)
#     can build a fresh estimator for each retrain window without repeating these arguments. ---
FINETUNE_LR         = 5e-4
FINETUNE_BATCH_SIZE = 64

def build_finetune_estimator(prediction_length, context_length, max_epochs):
    return LagLlamaEstimator(
        ckpt_path                = ckpt_path,
        prediction_length        = prediction_length,
        context_length           = context_length,
        input_size               = 1,
        n_layer                  = 8,
        n_embd_per_head          = 16,
        n_head                   = 9,
        scaling                  = "robust",
        num_parallel_samples     = 100,
        time_feat                = True,
        dropout                  = 0.0,
        device                   = device,
        nonnegative_pred_samples = False,   # FX/Gold/Oil returns can be negative
        aug_prob                 = 0,       # no synthetic augmentation -- fit OUR regime, not a generic one
        lr                       = FINETUNE_LR,
        batch_size               = FINETUNE_BATCH_SIZE,
        trainer_kwargs            = {"max_epochs": max_epochs},
    )

# --- Step 3: Directional-accuracy helpers (Requirement #3: replaces the old variance/MSE-based
#     confidence) + the dampening/cap transform (Requirement #4) ---
def directional_hitrate_on_window(predictor_, eval_df, asset_cols, macro_cols, scaler_,
                                   context_length, prediction_length, freq, step_days=21):
    """Coarse walk-forward check within eval_df: every `step_days` trading days, forecast
    `prediction_length` days ahead and check whether the forecast's direction (up/down vs. the
    last observed price) matches what actually happened. Returns {asset: hit_rate in [0,1]}.
    Uses the same context_length+10 history buffer used everywhere else in this notebook."""
    hist_window = context_length + 10
    hits = {a: [] for a in asset_cols}
    i = hist_window
    while i + prediction_length < len(eval_df):
        history = eval_df.iloc[max(0, i - hist_window): i]
        macro_scaled = scaler_.transform(history[macro_cols]).T.astype(np.float32)
        window_data = ListDataset(
            [{"start": history.index[0], "target": history[c].values.astype(np.float32),
              "feat_dynamic_real": macro_scaled} for c in asset_cols],
            freq=freq,
        )
        forecasts = list(predictor_.predict(window_data))
        last_row   = history.iloc[-1]
        future_row = eval_df.iloc[i + prediction_length]
        for c, forecast in zip(asset_cols, forecasts):
            pred_sign   = np.sign(float(forecast.mean.mean()) - last_row[c])
            actual_sign = np.sign(future_row[c] - last_row[c])
            if pred_sign != 0 and actual_sign != 0:
                hits[c].append(pred_sign == actual_sign)
        i += step_days
    return {a: (float(np.mean(v)) if len(v) > 0 else float('nan')) for a, v in hits.items()}


def confidence_from_hitrate(hit_rate):
    """Requirement #3: directional accuracy drives confidence instead of forecast variance/MSE.
    Requirement #4: global x0.5 dampening, then a hard cap at 0.75."""
    if hit_rate is None or np.isnan(hit_rate):
        return 0.25  # fallback only if a window had too few points to score
    raw = max(0.0, 2 * (hit_rate - 0.5))   # 0 at coin-flip skill (50%), 1 at perfect direction calls
    return float(min(raw * 0.5, 0.75))     # x0.5 dampening + hard cap at 0.75


def finetune_lag_llama(train_window_df, val_window_df, prediction_length, context_length,
                        asset_cols, macro_cols, freq, max_epochs):
    """Fine-tunes from the pretrained checkpoint on train_window_df, validates on val_window_df,
    and returns (predictor, {asset: confidence}, {asset: hit_rate}, fitted_scaler). Used both for
    the initial fine-tune below and for Cell 11's rolling re-training (Requirement #2)."""
    local_scaler = StandardScaler().fit(train_window_df[macro_cols])
    train_scaled = local_scaler.transform(train_window_df[macro_cols]).T.astype(np.float32)
    val_scaled   = local_scaler.transform(val_window_df[macro_cols]).T.astype(np.float32)

    train_ds = ListDataset(
        [{"start": train_window_df.index[0], "target": train_window_df[c].values.astype(np.float32),
          "feat_dynamic_real": train_scaled} for c in asset_cols], freq=freq)
    val_ds = ListDataset(
        [{"start": val_window_df.index[0], "target": val_window_df[c].values.astype(np.float32),
          "feat_dynamic_real": val_scaled} for c in asset_cols], freq=freq)

    est = build_finetune_estimator(prediction_length, context_length, max_epochs=max_epochs)
    predictor_ = est.train(train_ds, val_ds, cache_data=True, shuffle_buffer_length=1000)

    hit_rates = directional_hitrate_on_window(predictor_, val_window_df, asset_cols, macro_cols,
                                               local_scaler, context_length, prediction_length, freq)
    confidences = {a: confidence_from_hitrate(hr) for a, hr in hit_rates.items()}
    return predictor_, confidences, hit_rates, local_scaler


# --- Step 4: Historical monthly return std per asset (Requirement #5: computed from our own
#     2018-2024 data, not a hardcoded guess) -- used later to clip expected_return to +/-1.5x this ---
_hist_period = pd.concat([df_train, df_val]).set_index('Date')  # 2018-2024, same period we fine-tune on
HIST_MONTHLY_STD = {
    c: float(_hist_period[c].dropna().pct_change(21).dropna().std())  # ~21 trading days = 1 month
    for c in asset_cols
}
print("✅ Historical monthly return std per asset (2018-2024):")
for c, v in HIST_MONTHLY_STD.items():
    print(f"   {c}: {v:.4%}")

# --- Step 5: Initial fine-tune -- train on 2018-2022 (train_data), validate on 2023-2024
#     (val_data). Together that's "fine-tune on our 2018-2024 price data" (Requirement #1),
#     with 2023-2024 purely held out to monitor the fine-tune and score initial confidence. ---
INITIAL_MAX_EPOCHS = 30

print(f"\n⚙️  Fine-tuning Lag-Llama on 2018-2022, validating on 2023-2024 "
      f"({INITIAL_MAX_EPOCHS} epochs)...")
estimator = build_finetune_estimator(prediction_length, context_length, max_epochs=INITIAL_MAX_EPOCHS)
predictor = estimator.train(train_data, val_data, cache_data=True, shuffle_buffer_length=1000)
print("✅ Fine-tuned predictor ready (specialized to our assets, no longer zero-shot)!")

initial_hit_rates  = directional_hitrate_on_window(predictor, df_val.set_index('Date'), asset_cols,
                                                     macro_cols, scaler, context_length,
                                                     prediction_length, freq)
initial_confidence = {a: confidence_from_hitrate(hr) for a, hr in initial_hit_rates.items()}

print(f"   Prediction length : {prediction_length} days")
print(f"   Context length    : {context_length} trading days")
print(f"   Fine-tune epochs  : {INITIAL_MAX_EPOCHS}")
print(f"   Learning rate     : {FINETUNE_LR}")
print("   Directional hit-rate on 2023-2024 validation (per asset):")
for c in asset_cols:
    print(f"      {c}: hit-rate={initial_hit_rates[c]:.2%} -> confidence={initial_confidence[c]:.3f}")


In [ ]:
# ============================================================
# Cell 10 — Train/Val/Test Metrics for the Fine-Tuned Predictor
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# NOTE: `predictor` here is the FINE-TUNED predictor built in Cell 9 (Requirement #1) -- .train()
# already returns a ready-to-use predictor, so there's no separate "create predictor" step anymore.

# --- Step 1: Forecast on Train, Val, Test ---
print("🔮 Generating forecasts...")
train_forecasts = list(predictor.predict(train_data))
print("✅ Train forecasts generated!")
val_forecasts   = list(predictor.predict(val_data))
print("✅ Validation forecasts generated!")
test_forecasts  = list(predictor.predict(test_data))
print("✅ Test forecasts generated!")

# --- Step 2: Calculate Metrics (NaN safe) ---
def calc_metrics(forecasts, df_split):
    results = {}
    for col, forecast in zip(asset_cols, forecasts):
        mean_fc = forecast.mean

        # get actual values and drop NaN
        actual_raw = df_split[col].values
        actual_raw = actual_raw[~np.isnan(actual_raw)]  # remove NaN
        actual     = actual_raw[:prediction_length]      # take first 30

        # make sure lengths match
        min_len = min(len(actual), len(mean_fc))
        actual  = actual[:min_len]
        mean_fc = mean_fc[:min_len]

        mae  = mean_absolute_error(actual, mean_fc)
        rmse = np.sqrt(mean_squared_error(actual, mean_fc))
        mape = np.mean(np.abs((actual - mean_fc) / actual)) * 100

        results[col] = {
            "MAE" : round(mae, 4),
            "RMSE": round(rmse, 4),
            "MAPE": round(mape, 4),
        }
    return results

train_metrics = calc_metrics(train_forecasts, df_train)
val_metrics   = calc_metrics(val_forecasts,   df_val)
test_metrics  = calc_metrics(test_forecasts,  df_test)

# use test metrics for views
metrics = test_metrics

# --- Step 3: Print metrics comparison ---
print("\n📊 Metrics Comparison — Train vs Val vs Test:")
print("=" * 65)
for col in asset_cols:
    print(f"\n   📌 {col}")
    print(f"      {'Split':<12} {'MAE':>10} {'RMSE':>10} {'MAPE':>10}")
    print(f"      {'Train':<12} {train_metrics[col]['MAE']:>10} {train_metrics[col]['RMSE']:>10} {train_metrics[col]['MAPE']:>10}%")
    print(f"      {'Validation':<12} {val_metrics[col]['MAE']:>10} {val_metrics[col]['RMSE']:>10} {val_metrics[col]['MAPE']:>10}%")
    print(f"      {'Test':<12} {test_metrics[col]['MAE']:>10} {test_metrics[col]['RMSE']:>10} {test_metrics[col]['MAPE']:>10}%")
print("\n   (Directional accuracy -- the metric that now actually drives Confidence in Cell 11,")
print("   per Requirement #3 -- was reported above in Cell 9 for the 2023-2024 validation set.)")

# --- Step 4: Plot forecasts ---
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(asset_cols):
    history     = df_train[col].values[-60:]
    val_actual  = df_val[col].dropna().values[:prediction_length]
    val_fc      = val_forecasts[i].mean
    test_actual = df_test[col].dropna().values[:prediction_length]
    test_fc     = test_forecasts[i].mean
    train_fc    = train_forecasts[i].mean

    axes[i].plot(range(60), history,
                 label="Train Historical", color="blue")
    axes[i].plot(range(60, 60 + prediction_length),
                 train_fc, label="Train Forecast",
                 color="orange", marker="o", linestyle="--")
    axes[i].plot(range(60 + prediction_length,
                       60 + prediction_length * 2),
                 val_actual, label="Val Actual",
                 color="purple", marker="s")
    axes[i].plot(range(60 + prediction_length,
                       60 + prediction_length * 2),
                 val_fc, label="Val Forecast",
                 color="cyan", marker="o", linestyle="--")
    axes[i].plot(range(60 + prediction_length * 2,
                       60 + prediction_length * 3),
                 test_actual, label="Test Actual",
                 color="green", marker="s")
    axes[i].plot(range(60 + prediction_length * 2,
                       60 + prediction_length * 3),
                 test_fc, label="Test Forecast",
                 color="red", marker="o", linestyle="--")

    axes[i].set_title(
        f"{col}\n"
        f"Val MAPE={val_metrics[col]['MAPE']}% | "
        f"Test MAPE={test_metrics[col]['MAPE']}%"
    )
    axes[i].legend(fontsize=7)
    axes[i].grid(True)

plt.suptitle("Fine-Tuned Lag-Llama Forecast — Train vs Val vs Test",
             fontsize=13)
plt.tight_layout()
plt.show()

print("\n✅ Cell 10 Complete!")


## Cell 11 — Walk-forward views across ALL rebalance dates *(replaces the old single-snapshot version)*

**What changed and why:** the original version of this cell only extracted one forecast — a
single 5-day-ahead prediction made from the very start of the TEST period — giving 4 rows total
(one per asset) with no date attached. That's a single snapshot, not the time series Black-Litterman
needs at every monthly rebalance date.

This version instead loops across every monthly rebalance date in the TEST period (the same
cadence used everywhere else in the pipeline — `BL_model_with_WorldModel.ipynb`,
`Lag_Llama_WorldModel.ipynb`), and at each date:
1. builds a context window of the trailing ~64 trading days ending **at that date only** (no
   look-ahead — exactly like the original cell's context window, just re-anchored per date),
2. calls the same trained `predictor` used above,
3. records the forecast for that date and asset.

The output is now one row per `(Rebalance_Date, Asset)` pair — 19 dates × 4 assets = 76 rows —
instead of 4 rows with no date at all. Cells 1–10 above are unchanged; this is the only cell that
was modified.

**One thing this cell cannot verify in every environment:** if you're running this outside a
session with Hugging Face Hub access, `predictor.predict()` in Cell 9's checkpoint download will
have already failed before reaching this cell — that's an environment/network limitation, not a
bug in the loop below. The loop itself has been separately validated against a mock predictor with
the same interface (correct row count, correct dates, no missing values) — only the actual model
call needs a working `predictor` from Cells 8–9 to produce real numbers.


---
**Update — rolling re-training + confidence overhaul:** this cell's walk-forward loop now also:
1. periodically re-fine-tunes the model on an expanding window of data as it walks through the
   TEST period (Requirement #2), instead of using one fixed fine-tuned model for every date,
2. scores each fine-tuned model's own directional accuracy on its validation window and turns
   that into `Confidence`, replacing the old forecast-variance/MSE-based formula (Requirement #3),
3. applies a global x0.5 dampening and a hard 0.75 cap to every confidence score (Requirement #4),
4. keeps the +/-1.5x-historical-monthly-std clip on `Expected_Return` (Requirement #5), now
   computed from our actual 2018-2024 data in Cell 9 instead of a hardcoded dict.


In [ ]:
# ============================================================
# Cell 11 — Walk-Forward Lag-Llama Views for Black-Litterman (M1)
# Loops across every monthly rebalance date in TEST, with periodic rolling re-training
# ============================================================

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd

df_clean_idx = df_clean.set_index('Date')

# --- Step 1: monthly rebalance dates across TEST (version-robust, avoids the 'ME'/'M' pandas
#     alias mismatch some environments hit) ---
rebalance_dates = df_test.groupby(df_test['Date'].dt.to_period('M'))['Date'].max().tolist()
if df_test['Date'].iloc[0] not in rebalance_dates:
    rebalance_dates = [df_test['Date'].iloc[0]] + rebalance_dates
rebalance_dates = sorted(set(pd.Timestamp(d) for d in rebalance_dates))
rebalance_dates = [d for d in rebalance_dates if d in df_clean_idx.index]
print(f"📅 Forecasting at {len(rebalance_dates)} rebalance dates across the TEST period")

# --- Step 1b: Rolling re-training config (Requirement #2) -- re-fine-tunes on an EXPANDING
#     window (everything up to reb_date, no look-ahead) every ROLLING_RETRAIN_EVERY rebalance
#     dates, so the model keeps adapting to whatever regime the test period has moved into,
#     instead of using one fixed fine-tune (from Cell 9) for the whole 2025-2026 test period. ---
ROLLING_RETRAIN_EVERY = 3     # roughly quarterly, given the monthly rebalance cadence
ROLLING_MAX_EPOCHS    = 15    # shorter than the initial 30-epoch fit -- this is a refresh, not a
                               # from-scratch fit, since it still starts from lag-llama.ckpt each time
ROLLING_VAL_DAYS      = 252   # ~1 trading year held out as validation at each rolling retrain

current_predictor  = predictor            # fine-tuned in Cell 9
current_confidence = initial_confidence   # directional-accuracy confidence from Cell 9
current_hit_rates  = initial_hit_rates    # kept for the diagnostic column below
current_scaler     = scaler               # global macro scaler from Cell 7

# --- Step 2: walk-forward loop — one forecast call per rebalance date ---
all_rows = []
for i, reb_date in enumerate(rebalance_dates):

    if i > 0 and i % ROLLING_RETRAIN_EVERY == 0:
        window       = df_clean_idx.loc[:reb_date]           # everything up to (and incl.) reb_date
        val_window   = window.tail(ROLLING_VAL_DAYS)
        train_window = window.iloc[: -ROLLING_VAL_DAYS]
        print(f"   🔄 Rolling re-training at {reb_date.date()} on {len(train_window)} days "
              f"of history (validating on the most recent {ROLLING_VAL_DAYS} days)...")
        current_predictor, current_confidence, current_hit_rates, current_scaler = finetune_lag_llama(
            train_window, val_window, prediction_length, context_length,
            asset_cols, macro_cols, freq, max_epochs=ROLLING_MAX_EPOCHS)
        print("      updated confidence:",
              {a: round(c, 3) for a, c in current_confidence.items()})

    history = df_clean_idx.loc[:reb_date].tail(context_length + 10)
    if len(history) < context_length:
        print(f"   ⚠️  skipping {reb_date.date()} — not enough history yet")
        continue

    macro_hist_scaled = current_scaler.transform(history[macro_cols]).T.astype(np.float32)

    date_data = ListDataset(
        [
            {
                "start"            : history.index[0],
                "target"           : history[col].values.astype(np.float32),
                "feat_dynamic_real": macro_hist_scaled,
            }
            for col in asset_cols
        ],
        freq=freq,
    )

    date_forecasts = list(current_predictor.predict(date_data))

    for col, forecast in zip(asset_cols, date_forecasts):
        samples            = forecast.samples
        forecast_variance  = float(np.var(samples))   # kept as a diagnostic only -- NOT used
                                                        # for Confidence anymore (Requirement #3)
        mean_fc            = forecast.mean
        last_price         = float(history[col].values[-1])
        forecast_mean      = float(mean_fc.mean())
        expected_return    = (forecast_mean - last_price) / last_price if last_price else np.nan

        # Requirement #5: clip to +/- 1.5x historical monthly std per asset (HIST_MONTHLY_STD
        # computed from our own 2018-2024 data in Cell 9, not a hardcoded guess)
        er_limit         = 1.5 * HIST_MONTHLY_STD.get(col, 0.05)
        expected_return   = float(np.clip(expected_return, -er_limit, er_limit))
        signal            = "UP" if expected_return > 0 else "DOWN"

        # Requirements #3 + #4: confidence comes from the current fine-tuned model's own
        # directional accuracy (scored in Cell 9 initially, refreshed at each rolling retrain
        # above) -- NOT the old forecast-variance/MSE formula -- already x0.5-dampened and
        # capped at 0.75 by confidence_from_hitrate().
        confidence = current_confidence[col]

        hit_rate_val = current_hit_rates.get(col, float('nan'))
        if hit_rate_val is None or np.isnan(hit_rate_val):
            hit_rate_val = 0.5  # neutral fallback -- only hit if a validation window had too few points

        all_rows.append({
            "Rebalance_Date"               : reb_date.date(),
            "Asset"                        : col,
            "Last_Price"                   : round(last_price, 4),
            "Forecast_Mean"                : round(forecast_mean, 4),
            "Expected_Return"              : round(expected_return, 6),
            "Confidence"                   : round(confidence, 6),
            "Directional_HitRate"          : round(hit_rate_val, 4),
            "Forecast_Variance_Diagnostic" : round(forecast_variance, 6),
            "Signal"                       : signal,
        })

    print(f"   ✅ {reb_date.date()} — forecasted {len(asset_cols)} assets")

# --- Step 3: assemble, validate, and save ---
views_df = pd.DataFrame(all_rows)

assert views_df.shape[0] == len(rebalance_dates) * len(asset_cols), "row count must be dates x assets"
assert views_df.isna().sum().sum() == 0, "no missing values expected"
print(f"\n✅ Walk-forward forecasting complete: {views_df.shape[0]} rows "
      f"({views_df['Rebalance_Date'].nunique()} dates x {views_df['Asset'].nunique()} assets)")

views_df.to_csv("lag_llama_views.csv", index=False)
print("✅ Saved: lag_llama_views.csv")
views_df.head(8)
